## New Optimization Method : INCREMENTAL GRID SEARCH OPTIMIZATION (IGSO)

> *Authors* : Sasha Annabel, Kreshnayogi Dava Berliansyach

> *Description* : A new creative outlook of neighborhood search method using with adaptive step sizing.

### 1. IGSO Algorithm Functions (N-Dimensional)

In [ ]:
import itertools

# Updates step size according to change in function value (delta_f)
# Matches website-prototype parameters: expandMultiplier, shrinkMultiplier, minStep, maxStep
def update_step(delta_f, step, threshold, expand_multiplier=1.2, shrink_multiplier=0.8, min_step=0.01, max_step=3.0):
    if delta_f > threshold:
        next_step = min(step * expand_multiplier, max_step)
    else:
        next_step = max(step * shrink_multiplier, min_step)
    return next_step

In [ ]:
# Calculates value of objective function to be minimized
# Default trial function: 2D paraboloid with minimum at (2, 3)
def function(x, y):
    func = (x - 2)**2 + (y - 3)**2
    return func

In [ ]:
# Generates (3^d - 1) grid neighbors around current main node for any d-dimensional space
# For 2 variables (x, y): samples 3^2 - 1 = 8 neighbors
# For 4 variables (w, x, y, z): samples 3^4 - 1 = 80 neighbors
def grid_search(coords, step, threshold, func=function, expand_multiplier=1.2, shrink_multiplier=0.8, min_step=0.01, max_step=3.0):
    current_f = func(*coords)
    formatted_coords = ", ".join(f"{c:.4f}" for c in coords)
    print(f"Current Node: ({formatted_coords}) | f = {current_f:.5f}\n")

    best_coords = coords
    best_f = current_f

    # Generate grid offsets {-step, 0, +step} across all d dimensions
    d = len(coords)
    offsets = [-step, 0.0, step]

    # Evaluates all (3^d - 1) neighbors and the main node
    for delta in itertools.product(offsets, repeat=d):
        candidate = tuple(c + s for c, s in zip(coords, delta))
        check_f = func(*candidate)
        if check_f < best_f:
            best_f = check_f
            best_coords = candidate

    # Calculates improvement delta_f and updates step size
    delta_f = abs(current_f - best_f)
    action = "EXPAND" if delta_f > threshold else "SHRINK"
    formatted_best = ", ".join(f"{c:.4f}" for c in best_coords)
    print(f"Best Node: ({formatted_best}) | f = {best_f:.5f}")
    print(f"Improvement Δf: {delta_f:.5f} ({action})")

    step = update_step(delta_f, step, threshold, expand_multiplier, shrink_multiplier, min_step, max_step)
    return best_coords, step

### 2. Parameter Initialization
Consistent with the hyperparameters used in `website-prototype.html`.

In [ ]:
# Hyperparameter Initialization
start_coords = (5.0, 5.0)  # Starting position (supports any d variables, e.g. (5.0, 5.0, 5.0, 5.0))
threshold = 0.10
initial_step = 0.50
min_step = 0.01
max_step = 3.00
expand_multiplier = 1.20
shrink_multiplier = 0.80
tolerance = 0.0001
max_iterations = 60

### 3. Running Example

A trial program executing IGSO on **f(x,y) = (x-2)² + (y-3)²** to verify convergence to the analytical minimum (2, 3).

In [ ]:
coords = start_coords
step = initial_step
iteration = 1
current_f = function(*coords)

print(f"Initial Position: {coords} | Initial Step: {step:.4f} | f = {current_f:.5f}")

while current_f > tolerance and step > min_step and iteration <= max_iterations:
    print(f"\n{'='*20} Iteration {iteration} {'='*20}")
    coords, step = grid_search(coords, step, threshold, func=function, expand_multiplier=expand_multiplier, shrink_multiplier=shrink_multiplier, min_step=min_step, max_step=max_step)
    current_f = function(*coords)
    iteration += 1

print("\n" + "="*50)
print(f"Search Finished at Iteration {iteration - 1}")
formatted_final = ", ".join(f"{c:.4f}" for c in coords)
print(f"Final Node: ({formatted_final})")
print(f"Final Step Size: {step:.5f}")
print(f"Final f value: {current_f:.7f}")